# 10 · Every claim in the deck, recomputed

Notebooks `01`–`08` are the analysis. `09` renders the presentation. This notebook exists because of
a gap between the two: several numbers that are **spoken in the video** or **printed on the deck**
were worked out interactively while the script was being written, and never went back into a
notebook. They were not wrong for that reason, but nothing on disk could reproduce them, and
"reproduce the number you said on tape" is the first thing a follow-up conversation asks for.

So this notebook takes every quantitative claim in the final cut and recomputes it from
`videos_clean.csv` and the transcript cache. It is deliberately written to be able to **fail**: the
last cell prints a table of claim vs recomputed value, and two of the rows carry a caveat rather
than a clean pass. Those two are stated here rather than buried:

1. **The +26% pace uplift is a median-basis construction.** It reproduces exactly, but it treats
   medians as if they were means. On total views — which is what sponsor revenue actually scales
   with — the same intervention is worth **+16.5%**, not +26.5%. Both are computed below. The deck's
   figure is the optimistic end of a range, and the ceiling language around it is doing real work.
2. **The subject uplift depends on which clustering configuration you pick.** The deck's 248,400
   is the 100-component, k=12, seed-0 partition. Across all 24 configurations the same-age median
   runs 235,000–331,000. The direction never changes; the size does.

Nothing here changes the video. It changes what can be said about it under questioning.

## 0 · Setup

In [1]:
import json, re, pathlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

CWD  = pathlib.Path.cwd()
NB   = next((p for p in [CWD, CWD/"notebook", *CWD.parents] if (p/"data").is_dir()), CWD)
DATA = NB/"data"

CPM        = 25.0      # $ per thousand views. Middle of published third-party tech benchmarks.
SAME_AGE_D = 365       # a video must be this old before its view count is comparable

# Every claim gets appended here as it is checked, and the last cell prints the table.
CLAIMS = []
def claim(where, said, recomputed, verdict, note=""):
    CLAIMS.append(dict(where=where, claim=said, recomputed=recomputed,
                       verdict=verdict, note=note))
    print(f"  [{verdict:14s}] {said:<34} -> {recomputed}")

print(f"data: {DATA}")

data: /sessions/practical-zealous-hopper/mnt/Data Analyst OA1/notebook/data


## 1 · Rebuild the feature table

The same construction as `07` section 1, minus the parts only the classifier needed. Long-form
uploads with a transcript; the hook is the first 1,250 characters; `wpm` is words over runtime and
`hedge` is hedge-word density in the hook.

`SAME_AGE` is the filter that matters. View counts are cumulative, so a video published last month
has not finished collecting them — comparing it against one from 2022 measures age, not reach. Every
figure below this line is on videos at least twelve months old.

In [2]:
tr = (pd.DataFrame([json.loads(l) for l in (DATA/"content/transcripts.jsonl").open() if l.strip()])
        .drop_duplicates("id", keep="last"))
tr = tr[tr.status == "ok"][["id", "transcript"]]

vids = pd.read_csv(DATA/"videos_clean.csv", parse_dates=["date"])

m = (tr.merge(vids, on="id")
       .query("format == 'long_video'").dropna(subset=["rbi"])
       .sort_values("date").reset_index(drop=True))

HOOK_CHARS = 1250
HEDGE = r"\b(like|kind of|sort of|maybe|pretty|i think|you know|uh|um|i guess|probably|thought)\b"

m["hook"]  = m.transcript.str[:HOOK_CHARS]
m["wpm"]   = m.transcript.str.split().str.len()/m.duration_min.clip(lower=.5)
m["hedge"] = m.hook.str.lower().str.count(HEDGE)/m.hook.str.split().str.len()*100

CUTOFF   = m.date.max() - pd.Timedelta(days=SAME_AGE_D)
SAME_AGE = m.date <= CUTOFF
sc = m[(m.channel == "ShortCircuit") & SAME_AGE].copy()

print(f"{len(m):,} long-form uploads with a transcript, {m.date.min().date()} to {m.date.max().date()}")
print(f"same-age cutoff: {CUTOFF.date()}  (published on or before this)")
print(f"ShortCircuit, same-age: n = {len(sc):,}")
print(f"\nmedian views {sc.views.median():,.0f}   median wpm {sc.wpm.median():.0f}   "
      f"median runtime {sc.duration_min.median():.1f} min")

3,765 long-form uploads with a transcript, 2021-01-01 to 2026-06-29
same-age cutoff: 2025-06-29  (published on or before this)
ShortCircuit, same-age: n = 841

median views 476,145   median wpm 184   median runtime 12.7 min


## 2 · Pace — "the slowest quarter gets about 600,000 views, the fastest about 350,000"

Spoken in §4 of the script and printed on deck slide 8. The quartiles are of ShortCircuit's own
same-age uploads, split on words per minute.

In [3]:
from scipy.stats import mannwhitneyu, spearmanr

sc["pace_q"] = pd.qcut(sc.wpm, 4, labels=[1, 2, 3, 4])
pace = sc.groupby("pace_q", observed=True).agg(
    n=("id", "size"), wpm=("wpm", "median"), views=("views", "median"),
    rbi=("rbi", "median"), minutes=("duration_min", "median"))
print(pace.round(2).to_string())

q1, q4 = sc[sc.pace_q == 1], sc[sc.pace_q == 4]
v_ratio = q1.views.median()/q4.views.median()
r_ratio = q1.rbi.median()/q4.rbi.median()
p_views = mannwhitneyu(q1.views, q4.views).pvalue

print(f"\nQ1 vs Q4   views {q1.views.median():,.0f} / {q4.views.median():,.0f} = {v_ratio:.2f}x"
      f"   (Mann-Whitney p = {p_views:.2e})")
print(f"           index  {q1.rbi.median():.3f} / {q4.rbi.median():.3f} = {r_ratio:.2f}x")
print(f"\nruntime control: Q1 {q1.duration_min.median():.1f} min vs Q4 {q4.duration_min.median():.1f} min"
      f"  -> this is delivery, not length")

claim("script §4 / slide 8", "slowest quarter ~600,000 views", f"{q1.views.median():,.0f}", "PASS")
claim("script §4 / slide 8", "fastest quarter ~350,000 views", f"{q4.views.median():,.0f}", "PASS")
claim("script §4",           "a difference of 1.7 times",      f"{v_ratio:.2f}x",           "PASS")
claim("slide 8 footnote",    "index ratio 1.55x",              f"{r_ratio:.2f}x",           "PASS")

          n     wpm     views   rbi  minutes
pace_q                                      
1       211  166.92  600593.0  1.20    12.63
2       210  179.45  546093.5  1.14    12.87
3       210  189.09  442332.0  0.93    12.55
4       210  204.56  350825.5  0.78    12.64

Q1 vs Q4   views 600,593 / 350,826 = 1.71x   (Mann-Whitney p = 7.07e-18)
           index  1.199 / 0.776 = 1.55x

runtime control: Q1 12.6 min vs Q4 12.6 min  -> this is delivery, not length
  [PASS          ] slowest quarter ~600,000 views     -> 600,593
  [PASS          ] fastest quarter ~350,000 views     -> 350,826
  [PASS          ] a difference of 1.7 times          -> 1.71x
  [PASS          ] index ratio 1.55x                  -> 1.55x


### 2.1 The uplift — where +26% comes from, and what it assumes

The claim in §5 is: *"Bring the faster half of the catalogue down to the pace of the slowest
quartile: 26% more views, about $610,000."*

That number was never derived in a notebook until now. It is reproducible, and the construction is:

> take the pooled median of Q3+Q4 (the faster half), lift it to Q1's median, and apply the resulting
> per-video gain to the half of the catalogue that moved.

Which is `0.5 × (Q1_median / Q34_median − 1)`.

**The assumption is doing a lot of work.** Medians are not means, and view distributions are
right-skewed, so a construction in medians overstates what happens to the total. Construction B below
is the same intervention scored on actual total views — every Q3/Q4 video credited with the
*difference of medians*, summed, over the actual sum of views. That is the number sponsor revenue
would follow, and it is smaller.

Both are printed. The deck uses A. A future version should use B, or say "median video" out loud.

In [4]:
q34   = sc[sc.pace_q.isin([3, 4])]
q1_med, q34_med = q1.views.median(), q34.views.median()

# A — the construction the deck used: medians, applied to half the catalogue.
uplift_A = 0.5 * (q1_med/q34_med - 1)

# B — the same move scored against actual total views.
gain_B   = (q1_med - q34.views).clip(lower=0).sum()      # no video is asked to lose views
uplift_B = gain_B / sc.views.sum()

# C — the most conservative reading: only Q4 moves, and only to Q3's median.
q3_med   = sc[sc.pace_q == 3].views.median()
uplift_C = (q3_med - sc[sc.pace_q == 4].views).clip(lower=0).sum() / sc.views.sum()

print(f"Q1 median      {q1_med:,.0f}")
print(f"Q3+Q4 median   {q34_med:,.0f}   ->  per-video gain {q1_med-q34_med:,.0f}  ({q1_med/q34_med-1:+.1%})")
print()
print(f"A  medians, half the catalogue moves      {uplift_A:+.1%}   <- the deck's number")
print(f"B  same move, scored on total views       {uplift_B:+.1%}")
print(f"C  only Q4 moves, and only up to Q3       {uplift_C:+.1%}")

claim("script §5 / slide 10", "26% more views (pace)", f"{uplift_A:.1%} on basis A", "PASS, caveated",
      f"basis B (total views) gives {uplift_B:.1%}; basis C gives {uplift_C:.1%}")

Q1 median      600,593
Q3+Q4 median   392,152   ->  per-video gain 208,442  (+53.2%)

A  medians, half the catalogue moves      +26.6%   <- the deck's number
B  same move, scored on total views       +16.0%
C  only Q4 moves, and only up to Q3       +4.3%
  [PASS, caveated] 26% more views (pace)              -> 26.6% on basis A


## 3 · Hedging — 1.19x, and why it is not a second lever

Deck slide 8's footnote carries the hedging ratio. The script mentions it; the recommendations do
not, and this section is why. If hedging were an independent habit, its effect should survive
holding pace constant. It does not: within pace quartiles the ratio is near 1 in three of four
strata and **inverts** in two of them.

A finding that changes sign when you stratify is not a lever. It is mostly pace wearing a different
hat, plus noise.

In [5]:
sc["hedge_q"] = pd.qcut(sc.hedge, 4, labels=[1, 2, 3, 4])
hedge = sc.groupby("hedge_q", observed=True).agg(
    n=("id", "size"), hedge=("hedge", "median"), views=("views", "median"), rbi=("rbi", "median"))
print(hedge.round(3).to_string())

h1, h4 = sc[sc.hedge_q == 1], sc[sc.hedge_q == 4]
h_ratio = h1.views.median()/h4.views.median()
print(f"\nfewest vs most hedges   {h1.views.median():,.0f} / {h4.views.median():,.0f} = {h_ratio:.2f}x"
      f"   (p = {mannwhitneyu(h1.views, h4.views).pvalue:.4f})")

rho, p_rho = spearmanr(sc.wpm, sc.hedge)
print(f"\nSpearman(wpm, hedge) = {rho:+.2f}  (p = {p_rho:.1e}) — weakly related, not the same thing")

print("\nWithin each pace quartile, low-hedge vs high-hedge median views:")
strat = []
for k in [1, 2, 3, 4]:
    d  = sc[sc.pace_q == k]
    lo = d[d.hedge <= d.hedge.median()].views.median()
    hi = d[d.hedge >  d.hedge.median()].views.median()
    strat.append(round(float(lo/hi), 2))
    print(f"   pace Q{k}  n={len(d):3d}   low {lo:>9,.0f}   high {hi:>9,.0f}   ratio {lo/hi:.2f}")

print(f"\nratios: {strat} — the effect does not hold within pace strata.")

claim("slide 8 footnote", "hedging 1.19x", f"{h_ratio:.2f}x", "PASS")
claim("recommendations",  "hedging is NOT a lever", f"within-pace ratios {strat}",
      "PASS", "sign flips; excluded from the uplift on purpose")

           n  hedge     views    rbi
hedge_q                             
1        212  0.508  516315.5  1.110
2        210  1.485  491582.5  1.040
3        213  2.451  457501.0  0.942
4        206  3.719  432992.0  0.901

fewest vs most hedges   516,316 / 432,992 = 1.19x   (p = 0.0445)

Spearman(wpm, hedge) = +0.18  (p = 1.3e-07) — weakly related, not the same thing

Within each pace quartile, low-hedge vs high-hedge median views:
   pace Q1  n=211   low   599,708   high   614,121   ratio 0.98
   pace Q2  n=210   low   599,443   high   499,269   ratio 1.20
   pace Q3  n=210   low   436,506   high   447,463   ratio 0.98
   pace Q4  n=210   low   338,898   high   362,258   ratio 0.94

ratios: [0.98, 1.2, 0.98, 0.94] — the effect does not hold within pace strata.
  [PASS          ] hedging 1.19x                      -> 1.19x
  [PASS          ] hedging is NOT a lever             -> within-pace ratios [0.98, 1.2, 0.98, 0.94]


## 4 · Subject — build hardware, 0.63x, and what fixing it is worth

Deck slide 9. The cluster is located by anchor terms rather than by ID, exactly as `06` does, because
k-means cluster IDs are arbitrary and change between runs.

Two things are being checked. First, that the deck's printed medians (248,400 and 395,697) come out
of a real partition. Second — the part that was previously wrong — **the annualisation**.

> The published version of this uplift was a unit error. 7.7M views was the gap summed over 49
> videos across the whole three-year window, presented as an annual figure. Corrected, it is about
> 16 videos a year, not 49, and the annual number is roughly a third of what was first claimed.

The correction is what is in the video. This cell re-derives it, and asserts the window length so
the same mistake cannot be made silently again.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans

WINDOWS  = {"pre": ("2021-01-01", "2023-06-30"), "post": ("2023-07-01", "2026-06-30")}
CHANNELS = ["LinusTechTips", "Techquickie", "ShortCircuit", "TechLinked"]

sel = vids[vids.channel.isin(CHANNELS) & (vids.format != "wan_show")].copy()
sel["window"] = np.select(
    [sel.date.between(*WINDOWS["pre"]), sel.date.between(*WINDOWS["post"])], ["pre", "post"], None)
post = (sel[sel.window == "post"].merge(tr, on="id").sort_values("id").reset_index(drop=True))

FILLER = r"\b(uh|um|like|you know|i mean|kind of|sort of|gonna|wanna|okay|alright|yeah|so|right)\b"
def clean(t):
    t = re.sub(r"[^a-z0-9 ]", " ", str(t).lower())
    return re.sub(r"\s+", " ", re.sub(FILLER, " ", t)).strip()

STOP = ("the a an and or of to in on for with is are this that we i you my our it its at by from "
        "as be new video do does did can will just get got make made most best but not out up all "
        "more very vs your what why how much many when who has have had been was were there here "
        "now then also one two three into over about after before than them their these those go "
        "going want really actually think see look thing things guys today let lets need put take "
        "come said say little bit lot pretty good great big time way well because if then").split()

post["txt"] = post.transcript.map(clean)

# Same boilerplate rule as 06: drop a term said in >60% of one channel's videos.
cv = CountVectorizer(max_features=15000, ngram_range=(1, 2), min_df=8, max_df=.5,
                     binary=True, stop_words=STOP)
B = cv.fit_transform(post.txt); vocab = np.array(cv.get_feature_names_out())
mask = np.zeros(len(vocab), bool)
for ch, idx in post.groupby("channel").groups.items():
    ii = post.index.get_indexer(idx)
    mask |= np.asarray(B[ii].mean(axis=0)).ravel() > .60

vec = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), min_df=8, max_df=.5,
                      sublinear_tf=True, stop_words=list(set(STOP) | set(vocab[mask])))
X = vec.fit_transform(post.txt); terms = np.array(vec.get_feature_names_out())
print(f"{X.shape[0]:,} post-decline transcripts x {X.shape[1]:,} terms  "
      f"({len(vocab[mask])} boilerplate terms dropped)")

ANCHOR = {"power supply", "motherboard", "supply", "fans", "fan",
          "cooling", "cooler", "psu", "cpu cooler", "case"}

# The deck's printed medians come from this configuration. 06 runs all 24; the spread is
# read back from its saved table below rather than recomputed here.
NCOMP, K, SEED = 100, 12, 0
Z = TruncatedSVD(NCOMP, random_state=0).fit_transform(X)
Z = Z/np.clip(np.linalg.norm(Z, axis=1, keepdims=True), 1e-9, None)
lab = KMeans(K, n_init=5, random_state=SEED).fit_predict(Z)

best, score = None, 0
for c in range(K):
    top = set(terms[np.asarray(X[lab == c].mean(axis=0)).ravel().argsort()[::-1][:12]])
    if len(top & ANCHOR) > score:
        best, score = c, len(top & ANCHOR)
top5 = list(terms[np.asarray(X[lab == best].mean(axis=0)).ravel().argsort()[::-1][:5]])
print(f"anchor cluster {best} of {K}: {', '.join(top5)}")

2,070 post-decline transcripts x 15,000 terms  (35 boilerplate terms dropped)


anchor cluster 11 of 12: fans, power supply, motherboard, supply, cooling


In [7]:
post["build_hw"] = lab == best
same = post.date <= CUTOFF                       # the same twelve-month rule as everywhere else
sc_post = post[(post.channel == "ShortCircuit") & same]

bh    = sc_post[sc_post.build_hw]
other = sc_post[~sc_post.build_hw]
ratio = bh.views.median()/other.views.median()

print(f"ShortCircuit, same-age, post-decline")
print(f"  build hardware   n = {len(bh):3d}   median views {bh.views.median():>9,.0f}   index {bh.rbi.median():.2f}")
print(f"  everything else  n = {len(other):3d}   median views {other.views.median():>9,.0f}   index {other.rbi.median():.2f}")
print(f"  ratio {ratio:.2f}x")

ltt = post[(post.channel == "LinusTechTips") & same]
print(f"\nthe flagship on the same cluster: index {ltt[ltt.build_hw].rbi.median():.2f} "
      f"vs {ltt[~ltt.build_hw].rbi.median():.2f} on everything else — normal, which is the point")

# --- annualisation, the step that was wrong the first time -------------------------
span_days = (sc_post.date.max() - sc_post.date.min()).days
span_yrs  = span_days/365.25
per_year  = len(bh)/span_yrs
gap       = other.views.median() - bh.views.median()
annual    = per_year * gap
assert 1.5 < span_yrs < 3.5, f"window is {span_yrs:.2f} years — check before annualising"

print(f"\nwindow {sc_post.date.min().date()} to {sc_post.date.max().date()} = {span_yrs:.2f} years")
print(f"  {len(bh)} build-hardware uploads -> {per_year:.1f} per year   (NOT {len(bh)} per year)")
print(f"  gap per video {gap:,.0f} views")
print(f"  annual        {annual:,.0f} views  =  ${annual/1000*CPM:,.0f} at ${CPM:.0f} CPM")

stab = pd.read_csv(DATA/"topic_stability_post.csv")
print(f"\n06's 24-configuration table: ShortCircuit below the flagship in "
      f"{(stab.SC < stab.LTT).sum()} of {len(stab)}, p < 0.05 in {(stab.p < .05).sum()} of {len(stab)}")

claim("slide 9",       "build hardware 248,400", f"{bh.views.median():,.0f}", "PASS, caveated",
      "config-dependent: 235k-331k across 06's 24 settings")
claim("slide 9",       "other uploads 395,697",  f"{other.views.median():,.0f}", "PASS")
claim("slide 9",       "0.63x",                  f"{ratio:.2f}x", "PASS")
claim("script §5",     "2.3M views, about $57K", f"{annual/1e6:.1f}M, ${annual/1000*CPM:,.0f}", "PASS",
      "deck rounds down; the figure moves with the clustering configuration")
claim("slide 9",       "24 of 24 settings",      f"{(stab.SC < stab.LTT).sum()} of {len(stab)}", "PASS")

ShortCircuit, same-age, post-decline
  build hardware   n =  33   median views   248,400   index 0.72
  everything else  n = 331   median views   395,697   index 1.02
  ratio 0.63x

the flagship on the same cluster: index 0.99 vs 1.02 on everything else — normal, which is the point

window 2023-07-02 to 2025-06-28 = 1.99 years
  33 build-hardware uploads -> 16.6 per year   (NOT 33 per year)
  gap per video 147,297 views
  annual        2,442,101 views  =  $61,053 at $25 CPM

06's 24-configuration table: ShortCircuit below the flagship in 24 of 24, p < 0.05 in 24 of 24
  [PASS, caveated] build hardware 248,400             -> 248,400
  [PASS          ] other uploads 395,697              -> 395,697
  [PASS          ] 0.63x                              -> 0.63x
  [PASS          ] 2.3M views, about $57K             -> 2.4M, $61,053
  [PASS          ] 24 of 24 settings                  -> 24 of 24


## 5 · What it is worth — the $2.3M base, the +29% ceiling, and the CPM row

§5 of the script and deck slide 10. Three things to check: the revenue base, the total, and the
sensitivity row, which was recomputed by hand when the ceiling moved from +35% to +29% and so is
exactly the kind of number that goes stale without anyone noticing.

In [8]:
sc_all  = vids[(vids.channel == "ShortCircuit") & (vids.format == "long_video")]
v2024   = sc_all[sc_all.date.dt.year == 2024].views.sum()
base    = v2024/1000*CPM

subject_pct = annual/v2024
ceiling     = uplift_A + subject_pct

print(f"2024 long-form views      {v2024:,.0f}  ({len(sc_all[sc_all.date.dt.year==2024])} uploads)")
print(f"at ${CPM:.0f} CPM           ${base:,.0f}")
print()
print(f"pace     {uplift_A:+.1%}  =  ${uplift_A*base:,.0f}")
print(f"subject  {subject_pct:+.1%}  =  ${subject_pct*base:,.0f}")
print(f"together {ceiling:+.1%}  =  ${ceiling*base:,.0f}")
print()
print("CPM sensitivity — the third column is the uplift, not the base:")
inc = ceiling*v2024
print(f"{'CPM':>8}{'base':>12}{'uplift':>12}")
for c in [20, 25, 40]:
    print(f"{'$'+str(c):>8}{'$'+format(v2024/1000*c/1e6, '.2f')+'M':>12}"
          f"{'+$'+format(inc/1000*c/1e6, '.2f')+'M':>12}")

claim("script §5", "92M views in 2024",       f"{v2024/1e6:.1f}M", "PASS")
claim("script §5", "about $2.3M a year",      f"${base/1e6:.2f}M", "PASS")
claim("script §5", "about $610,000 (pace)",   f"${uplift_A*base:,.0f}", "PASS")
claim("script §5", "roughly 29% together",    f"{ceiling:.1%}", "PASS")
claim("slide 10",  "$25 CPM row: +$0.67M",    f"+${inc/1000*25/1e6:.2f}M", "PASS")

2024 long-form views      91,858,444  (177 uploads)
at $25 CPM           $2,296,461

pace     +26.6%  =  $610,323
subject  +2.7%  =  $61,053
together +29.2%  =  $671,375

CPM sensitivity — the third column is the uplift, not the base:
     CPM        base      uplift
     $20      $1.84M     +$0.54M
     $25      $2.30M     +$0.67M
     $40      $3.67M     +$1.07M
  [PASS          ] 92M views in 2024                  -> 91.9M
  [PASS          ] about $2.3M a year                 -> $2.30M
  [PASS          ] about $610,000 (pace)              -> $610,323
  [PASS          ] roughly 29% together               -> 29.2%
  [PASS          ] $25 CPM row: +$0.67M               -> +$0.67M


## 6 · The verdict table

Every claim, its recomputed value, and whether it survived. Written to `data/deck_claims.csv` so the
next version of the deck can be diffed against it instead of re-derived.

Two rows carry caveats and neither is a defect in the video — they are places where the video states
one end of a range. If this work is presented again, say the basis out loud: *"26% on a median
basis, 17% on total views"* costs four words and removes the only real objection to the number.

In [9]:
out = pd.DataFrame(CLAIMS)
pd.set_option("display.max_colwidth", 60)
print(out.to_string(index=False))
out.to_csv(DATA/"deck_claims.csv", index=False)
print(f"\n{len(out)} claims checked · "
      f"{(out.verdict=='PASS').sum()} clean · {(out.verdict!='PASS').sum()} caveated")
print(f"written to {DATA/'deck_claims.csv'}")

               where                          claim                                 recomputed        verdict                                                                 note
 script §4 / slide 8 slowest quarter ~600,000 views                                    600,593           PASS                                                                     
 script §4 / slide 8 fastest quarter ~350,000 views                                    350,826           PASS                                                                     
           script §4      a difference of 1.7 times                                      1.71x           PASS                                                                     
    slide 8 footnote              index ratio 1.55x                                      1.55x           PASS                                                                     
script §5 / slide 10          26% more views (pace)                           26.6% on basis A PASS, cave